# Technical Note P7: SciPy Linear Algebra

**Wook Kang · Revised English edition · Version 1.0.0 · 3 October 2026**

This note connects a mathematical task to a numerical object, an implementation, and evidence about the result. It is an instructional computational document. Demonstration parameters are illustrative unless explicitly stated otherwise.

Run from a fresh kernel in cell order. See [the series README](../README.md) for setup and [editorial and verification notes](../docs/EDITORIAL_NOTES.md) for scope and limitations. The English prose has been reorganized and expanded from the supplied Korean notebook; this is an adapted edition rather than a line-by-line translation.

**Reading question:** What is given, what is unknown, what operation relates them, and what independent check can assess the output?

## Core mathematical structure

$$
\boxed{
\text{problem structure}
\rightarrow
\text{appropriate linear algebra algorithm}
}
$$

$$
\boxed{
Ax=b\ ?\qquad
Av=\lambda v\ ?\qquad
Ax\approx b\ ?
}
$$

In [1]:
# Reproducible display and scale-aware comparison.
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
def scaled_allclose(a, b, rtol=1e-12, atol=None, **kwargs):
    """Compare numerical examples using an absolute tolerance scaled to b."""
    a, b = np.asarray(a), np.asarray(b)
    scale = float(np.max(np.abs(b))) if b.size else 0.0
    if atol is None:
        atol = 1e-14 * max(scale, np.finfo(float).tiny)
    return np.allclose(a, b, rtol=rtol, atol=atol, **kwargs)


## 1. NumPy and SciPy linalg

NumPy supplies core dense operations; SciPy adds factorization interfaces, structured solves, and matrix functions. Select by mathematical structure and reuse needs.

| Purpose | NumPy | SciPy |  
| --- | --- | --- |  
| Basic dense solve | Supported | Supported |  
| eigen / SVD | Supported | Supported |  
| Detailed factorization control | Limited | More extensive |  
| generalized eigenproblem | Limited | Supported |  
| matrix functions | Limited | Extensive |  
| Access to LAPACK routines | Basic | More direct |

In [2]:
import numpy as np
from scipy import linalg

print("NumPy version:", np.__version__)

NumPy version: 2.3.5


## 2. The linear system Ax=b

A maps an unknown vector to a known right-hand side. Inspect shape, units, constraints, and residual after solving.

$$
\mathbf A\mathbf x=\mathbf b
$$

$$
\mathbf r=\mathbf A\mathbf x-\mathbf b
$$

In [3]:
A = np.array([
    [3.0, 1.0],
    [1.0, 2.0],
])
b = np.array([9.0, 8.0])

x = linalg.solve(A, b)
r = A @ x - b

print("x =", x)
print("residual =", r)
print("residual norm =", np.linalg.norm(r))

x = [2. 3.]
residual = [1.77635684e-15 0.00000000e+00]
residual norm = 1.7763568394002505e-15


## 3. Avoid explicit inversion

Solve the system directly rather than computing an inverse and multiplying. This usually avoids unnecessary work and additional rounding error.

$$
\mathbf x=\mathbf A^{-1}\mathbf b
$$

```python
solve(A,b)
```

In [4]:
x_solve = linalg.solve(A, b)
x_inv = linalg.inv(A) @ b

print("solve =", x_solve)
print("inv@b =", x_inv)
print("difference =", np.linalg.norm(x_solve - x_inv))

solve = [2. 3.]
inv@b = [2. 3.]
difference = 4.440892098500626e-16


## 4. LU factorization

Factor once and solve for multiple right-hand sides when A is unchanged. Check permutation conventions when reconstructing a library factorization.

$$
PA=LU
$$

$$
\boxed{
A\text{ fixed}
\Rightarrow
\text{factor once, solve many times}
}
$$

In [5]:
P, L, U = linalg.lu(A)

print("reconstruction error =", np.linalg.norm(A - P @ L @ U))

lu, piv = linalg.lu_factor(A)

b1 = np.array([9.0, 8.0])
b2 = np.array([1.0, 0.0])

x1 = linalg.lu_solve((lu, piv), b1)
x2 = linalg.lu_solve((lu, piv), b2)

print("x1 =", x1)
print("x2 =", x2)

reconstruction error = 0.0
x1 = [2. 3.]
x2 = [ 0.4 -0.2]


## 5. Cholesky factorization

Cholesky exploits a Hermitian positive-definite matrix. Symmetry alone is insufficient, and unused triangular entries should not be interpreted as part of the factor.

$$
A=LL^T
$$

$$
A=R^TR
$$

In [6]:
A_spd = np.array([
    [4.0, 1.0],
    [1.0, 3.0],
])

L = linalg.cholesky(A_spd, lower=True)
eigvals = linalg.eigvalsh(A_spd)

print("L =")
print(L)
print("reconstruction error =", np.linalg.norm(A_spd - L @ L.T))
print("eigenvalues =", eigvals)
print("positive definite?", np.all(eigvals > 0))

L =
[[2.        0.       ]
 [0.5       1.6583124]]
reconstruction error = 0.0
eigenvalues = [2.38196601 4.61803399]
positive definite? True


## 6. QR and least squares

QR supports rectangular least-squares problems without forming normal equations. Normal equations square the spectral condition number for full-column-rank matrices.

$$
A=QR
$$

$$
Ax\approx b
$$

$$
\min_x\|Ax-b\|_2^2
$$

$$
A^TAx=A^Tb
$$

In [7]:
A_ls = np.array([
    [1.0, 0.0],
    [1.0, 1.0],
    [1.0, 2.0],
    [1.0, 3.0],
])
b_ls = np.array([1.0, 2.1, 2.9, 4.2])

Q, R = linalg.qr(A_ls, mode="economic")
x_ls, residuals, rank, s = linalg.lstsq(A_ls, b_ls)

print("Q.T @ Q =")
print(Q.T @ Q)
print("x =", x_ls)
print("rank =", rank)
print("singular values =", s)

Q.T @ Q =
[[1.00000000e+00 5.55111512e-17]
 [5.55111512e-17 1.00000000e+00]]
x = [0.99 1.04]
rank = 2
singular values = [4.10003045 1.09075677]


## 7. Eigenproblems

Use a symmetric or Hermitian solver when that structure applies. Check eigenpair residuals and understand that eigenvector signs and repeated-eigenvalue bases are not unique.

$$
Av=\lambda v
$$

```python
eigh
eigvalsh
```

```python
eig
eigvals
```

In [8]:
A_eig = np.array([
    [2.0, 1.0],
    [1.0, 2.0],
])

w, V = linalg.eigh(A_eig)

print("eigenvalues =", w)
print("eigenvectors =")
print(V)

for i in range(len(w)):
    residual = A_eig @ V[:, i] - w[i]*V[:, i]
    print(f"eigenpair {i} residual norm =", np.linalg.norm(residual))

eigenvalues = [1. 3.]
eigenvectors =
[[-0.70710678  0.70710678]
 [ 0.70710678  0.70710678]]
eigenpair 0 residual norm = 0.0
eigenpair 1 residual norm = 0.0


## 8. Generalized eigenproblems

K-phi=lambda-M-phi connects stiffness and mass operators. A symmetric generalized solver requires the appropriate positive-definite metric matrix.

$$
K\phi=\lambda M\phi
$$

In [9]:
K = np.array([
    [2.0, -1.0],
    [-1.0, 2.0],
])

M = np.array([
    [2.0, 0.0],
    [0.0, 1.0],
])

lam, modes = linalg.eigh(K, M)

print("lambda =", lam)
print("omega =", np.sqrt(lam))
print("modes =")
print(modes)

lambda = [0.6339746 2.3660254]
omega = [0.79622522 1.538189  ]
modes =
[[-0.62796303 -0.32505758]
 [-0.45970084  0.88807383]]


## 9. Singular value decomposition

SVD reveals amplification, rank deficiency, and poorly determined input directions. Singular vectors depend on scaling and the selected units.

$$
A=U\Sigma V^T
$$

In [10]:
A_svd = np.array([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0],
])

U, s, VT = linalg.svd(A_svd, full_matrices=False)

print("singular values =", s)
print("reconstruction error =", np.linalg.norm(A_svd - U @ np.diag(s) @ VT))

singular values = [9.52551809 0.51430058]
reconstruction error = 1.2755491433176288e-15


## 10. Rank and conditioning

Numerical rank uses a threshold. Report singular values or a condition number rather than relying solely on a binary rank decision.

$$
\kappa(A)=
\frac{\sigma_{\max}}{\sigma_{\min}}
$$

$$
\boxed{
\text{ill-conditioned}
\neq
\text{solver failure}
}
$$

In [11]:
A_good = np.array([
    [2.0, 0.0],
    [0.0, 1.0],
])

A_bad = np.array([
    [1.0, 0.999999],
    [0.999999, 0.9999981],
])

print("rank bad =", np.linalg.matrix_rank(A_bad))
print("cond good =", np.linalg.cond(A_good))
print("cond bad  =", np.linalg.cond(A_bad))
print("singular values bad =", linalg.svdvals(A_bad))

rank bad = 2
cond good = 2.0
cond bad  = 40000322.05362146
singular values bad = [1.99999805e+00 4.99995487e-08]


## 11. Perturbation amplification

Compare changes in input and solution to expose sensitivity. A small residual can coexist with a large solution error in an ill-conditioned system.

In [12]:
A = np.array([
    [1.0, 1.0],
    [1.0, 1.000001],
])

b1 = np.array([2.0, 2.000001])
b2 = b1 + np.array([0.0, 1e-7])

x1 = linalg.solve(A, b1)
x2 = linalg.solve(A, b2)

print("x1 =", x1)
print("x2 =", x2)
print("RHS change =", np.linalg.norm(b2-b1))
print("solution change =", np.linalg.norm(x2-x1))

x1 = [1. 1.]
x2 = [0.9 1.1]
RHS change = 9.999999983634211e-08
solution change = 0.1414213560174964


## 12. Determinant is not a conditioning test

Determinants change dramatically under scaling and dimension changes. Singular values provide a more useful numerical diagnosis.

$$
\det A=0
$$

## 13. Matrix functions

Expm and sqrtm compute matrix functions rather than elementwise exponentials or square roots. These operations have different meanings from NumPy ufuncs.

$$
e^A
$$

$$
\dot x=Ax
$$

$$
x(t)=e^{At}x(0)
$$

In [13]:
A = np.array([
    [0.0, 1.0],
    [-1.0, 0.0],
])

expA = linalg.expm(A)

print("exp(A) =")
print(expA)

B = np.diag([4.0, 9.0])
sqrtB = linalg.sqrtm(B)

print("\nsqrt(B) =")
print(sqrtB)

exp(A) =
[[ 0.54030231  0.84147098]
 [-0.84147098  0.54030231]]

sqrt(B) =
[[2. 0.]
 [0. 3.]]


## 14. Structure guides the algorithm

Dense, sparse, symmetric, positive-definite, banded, and rectangular systems need different strategies. Verify claimed structure before using a specialized solver.

## 15. PDE discretization leads to linear algebra

Unknown ordering and boundary conditions define the assembled system. Algebraic residuals assess the discrete equations, not the continuum approximation by themselves.

$$
Au=b
$$

$$
U_{ij}
\leftrightarrow
u_I
$$

$$
\boxed{
\text{PDE field}
\rightarrow
\text{global vector}
\rightarrow
\text{linear system}
}
$$

## 16. A small Poisson example

The positive tridiagonal operator represents minus-u-double-prime with homogeneous Dirichlet values. A quadratic exact solution is reproduced unusually well because the centred stencil is exact for quadratics.

$$
-\frac{d^2u}{dx^2}=1,
\qquad
u(0)=u(1)=0
$$

$$
-\frac{u_{i-1}-2u_i+u_{i+1}}{\Delta x^2}=1
$$

In [14]:
N = 8
L = 1.0
dx = L/(N+1)

A = np.zeros((N, N))
b = np.ones(N)

for i in range(N):
    A[i, i] = 2.0/dx**2
    if i > 0:
        A[i, i-1] = -1.0/dx**2
    if i < N-1:
        A[i, i+1] = -1.0/dx**2

u = linalg.solve(A, b)

x = np.linspace(dx, L-dx, N)
u_exact = 0.5*x*(1-x)

print("max error =", np.max(np.abs(u-u_exact)))
print("residual norm =", np.linalg.norm(A @ u - b))

max error = 2.7755575615628914e-17
residual norm = 0.0


$$
\boxed{
\text{dense linear algebra}
\rightarrow
\text{sparse linear algebra}
}
$$

## 17. Dense-solver map

Match solves, LU, Cholesky, QR, eigenproblems, SVD, and matrix functions to their mathematical tasks. Factorization reuse is distinct from rerunning a complete solve.

| Mathematical problem | SciPy |  
| --- | --- |  
| $Ax=b$ | `scipy.linalg.solve` |  
| repeated $Ax=b$ | `lu_factor`, `lu_solve` |  
| SPD solve | `cho_factor`, `cho_solve` |  
| triangular solve | `solve_triangular` |  
| $A=QR$ | `qr` |  
| $Av=\lambda v$ | `eig`, `eigh` |  
| $Kv=\lambda Mv$ | `eigh(K,M)` |  
| $Ax\approx b$ | `lstsq` |  
| SVD | `svd`, `svdvals` |  
| matrix exponential | `expm` |  
| matrix square root | `sqrtm` |

## 18. Continue to P8

Sparse storage and Krylov methods become important when the operator couples only nearby unknowns on a large mesh.

$$
\boxed{
\text{dense}
\rightarrow
\text{large-scale sparse}
}
$$

## Further work

1. Change one parameter or discretization choice and predict the effect before running.
2. Write the input and output shapes, units, and field locations.
3. Construct a deliberately invalid input and make the calculation report it clearly.
4. Separate an execution check from an accuracy check and from any physical claim.

## Primary documentation

- [NumPy linear algebra](https://numpy.org/doc/stable/reference/routines.linalg.html)
- [SciPy user guide](https://docs.scipy.org/doc/scipy/tutorial/)

These are living documentation links. The accompanying requirements and verification report identify the versions actually executed for this edition.

## AI assistance and responsibility

The English adaptation, editorial supplementation, and code review were prepared with AI assistance. Wook Kang retains responsibility for reviewing the explanations, examples, and any subsequent research application. No new experimental validation is claimed.